# meas_crowd with pipe_crowd's sparse matrix solver: speed and agreement test

Every joint flux fit in meas_crowd (all sources of a cutout fitted together, fluxes >= 0) now
goes through the solver from [lsst-dm/pipe_crowd](https://github.com/lsst-dm/pipe_crowd): the
sparse (pixels x sources) design matrix is solved with Eigen's `LeastSquaresConjugateGradient`
in C++, without ever forming the dense n x n normal matrix.  An active-set loop around it keeps
meas_crowd's non-negative fluxes.  It is selected with the `jointSolver` option:

| `jointSolver` | what it does |
| --- | --- |
| `"eigen"` (new default) | pipe_crowd's sparse Eigen LSCG + x >= 0 active set (C++) |
| `"normal"` (previous default) | dense normal-equation Cholesky + NNLS for >= 300 sources, `scipy.optimize.lsq_linear` below |
| `"scipy"` | `scipy.optimize.lsq_linear` (method 'trf') always |

This notebook

1. builds the C++ extension if needed;
2. times the solvers on synthetic crowded tiles (no data access needed);
3. fits the **same real LSSTCam cutout** with `"normal"` and `"eigen"` and compares the run time,
   the catalogues and the residuals.

Run the cells in order with the **LSST** kernel.

## 0. Set up

A Jupyter kernel does not see `setup -k -r .` from a terminal, so this cell puts the checkout on
the path itself.  Set `MEAS_CROWD` if the notebook is not in the checkout's `examples/` folder.

In [ ]:
%matplotlib inline
import os
import subprocess
import sys
import time

# where this repository is checked out (default: the folder above this notebook)
MEAS_CROWD = os.environ.get("MEAS_CROWD_DIR") or os.path.abspath("..")
# MEAS_CROWD = os.path.expanduser("~/meas_crowd_v2")
PKG = os.path.join(MEAS_CROWD, "python")
if not os.path.isdir(os.path.join(PKG, "lsst", "meas", "crowd")):
    raise FileNotFoundError(f"no meas_crowd package under {PKG}: set MEAS_CROWD above")
if PKG not in sys.path:
    sys.path.insert(0, PKG)
import lsst  # noqa: E402
import lsst.meas  # noqa: E402
for mod, sub in ((lsst, "lsst"), (lsst.meas, os.path.join("lsst", "meas"))):
    if os.path.join(PKG, sub) not in list(mod.__path__):   # lsst / lsst.meas were imported before
        mod.__path__.append(os.path.join(PKG, sub))

### Build the C++ extension (once)

The solver lives in the package's C++ extension (`_crowdLib`), next to the fast PSF stamps.  This
cell builds it in place with `setup.py` if it is missing or was built before the solver existed.
It needs pybind11 and the Eigen headers, both in the RSP's stack environment (`$EIGEN_DIR` or
`$CONDA_PREFIX/include/eigen3`).  **If it builds, restart the kernel and run from the top.**

(From a terminal, `setup lsst_distrib; cd <checkout>; setup -k -r .; scons` does the same.)

In [ ]:
import glob

CHECK = """
import importlib.util, sys
spec = importlib.util.spec_from_file_location("_crowdLib", sys.argv[1])
m = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m)
print(hasattr(m, "solveSparseLeastSquares"))
"""

def solverBuilt():
    """True if the compiled extension has the sparse solver (checked in a fresh process)."""
    for so in glob.glob(os.path.join(PKG, "lsst", "meas", "crowd", "_crowdLib*.so")):
        out = subprocess.run([sys.executable, "-c", CHECK, so], capture_output=True, text=True)
        if out.stdout.strip() == "True":
            return True
    return False

if solverBuilt():
    print("C++ extension with the sparse solver: OK")
else:
    print("building the C++ extension ...")
    r = subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"], cwd=MEAS_CROWD,
                       capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    if r.returncode == 0 and solverBuilt():
        print("\nBUILT -- now restart the kernel (Kernel > Restart) and run from the top.")
    else:
        raise RuntimeError("build failed: see the output above")

In [ ]:
import numpy as np  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
from scipy.spatial import cKDTree  # noqa: E402

import lsst.meas.crowd as crowd  # noqa: E402

print("meas_crowd from:", os.path.dirname(crowd.__file__))
print("C++ kernels:", crowd.HAVE_CXX, "| sparse Eigen solver:", crowd.HAVE_SPARSE_SOLVER)
assert crowd.HAVE_SPARSE_SOLVER, "the sparse solver is not built: run the build cell, restart"

# The notebook server's CPU limit is a quota (e.g. 4 CPUs) on a node whose cores are all visible
# (e.g. 128): without a cap, NumPy's BLAS starts a thread per visible core and the kernel is
# throttled.  Cap the threads at the usable CPUs.
crowd.limitBlasThreads()

## 1. Synthetic crowded tiles: solver time against the number of sources

`makeCrowdedFieldProblem` builds the same kind of matrix as the pipeline's joint fit
(`fitLib.jointPsfFitDiag`): a 460 x 460 px image (a 400 px tile with 30 px padding), an 81 x 81 px
PSF with wings, stamps cropped where a source's wing drops below 0.05 sigma per pixel, rows
weighted by 1/sigma.  `compareSolvers` times each solver on it and checks they agree.

The old solvers get slow fast (the dense path scales as n^3): `SLOW_LIMIT` caps the sizes they
are run on.

In [ ]:
SIZES = [300, 1000, 2000, 4000]
SLOW_LIMIT = {"normal": 4000, "scipy": 1000}     # skip the old solvers above these sizes

bench = {}
for n in SIZES:
    A, b, truth = crowd.makeCrowdedFieldProblem(nSources=n, seed=1)
    names = ["eigen"] + [s for s in ("normal", "scipy") if n <= SLOW_LIMIT[s]]
    bench[n] = crowd.compareSolvers(A, b, solvers=tuple(names))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
for name, mk in (("eigen", "o-"), ("normal", "s--"), ("scipy", "^:")):
    ns = [n for n in SIZES if name in bench[n]]
    ax.loglog(ns, [bench[n][name]["seconds"] for n in ns], mk, label=name)
ax.set_xlabel("sources in the joint fit")
ax.set_ylabel("seconds per joint fit")
ax.set_title("one joint flux fit, synthetic 460 px tile")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.show()
for n in SIZES:
    e = bench[n]["eigen"]
    others = ", ".join(f"{k}: {v['seconds'] / e['seconds']:.1f}x the eigen time, "
                       f"chi2 ratio {v['cost'] / e['cost']:.9f}"
                       for k, v in bench[n].items() if k != "eigen")
    print(f"{n:5d} sources: eigen {e['seconds']:.2f} s | {others}")

## 2. A real cutout, old solver against new

The same cutout is fitted twice, once with `jointSolver="normal"` (the previous behaviour) and
once with `"eigen"`; nothing else changes.  `crowd.solverStats()` counts the time spent inside
joint flux fits only, so the two numbers per run are *joint-fit time* and *total time*.

The fit is iterative and adaptive (sources are added, merged and removed on S/N thresholds), so
two solvers that agree to ~1e-6 can still tip an occasional borderline decision; expect the same
catalogue up to a handful of faint sources.

Pick a crowded cutout to see the difference: the gain grows with the number of sources per fit.

In [ ]:
VISIT, DETECTOR = 2025062100433, 144
CUTOUT = "1000,1000,300"            # x0, y0, size (detector px); try 400 for a full tile
PAD = 20
PSF_SOURCE = "visit"
REPO, COLLECTION = "dp2_prep", "LSSTCam/runs/DRP/DP2"
MAX_OUTER, MAX_INNER = 3, 15
SOLVERS = ["normal", "eigen"]       # old, new
OUT = "solver_test"
os.makedirs(OUT, exist_ok=True)

In [ ]:
from lsst.daf.butler import Butler  # noqa: E402

butler = Butler(REPO, collections=COLLECTION)
refs = butler.query_datasets("visit_image", collections="LSSTCam/runs/*", find_first=False,
                             where=f"instrument='LSSTCam' AND visit={VISIT} AND detector={DETECTOR}")
visitImage = butler.get(refs[0])

cfg = crowd.makeConfig(crowd.CROWDED_FIELD_CONFIG)
sess = crowd.prepareSession(VISIT, DETECTOR, [CUTOUT], pad=PAD, psfSource=PSF_SOURCE,
                             visitImage=visitImage)
crowd.attachGaia(sess, visitImage=visitImage, repo=REPO)      # cached in gaia_cache/

In [ ]:
runs, timing = {}, {}
for solver in SOLVERS:
    print(f"\n=================== jointSolver = {solver!r} ===================")
    crowd.solverStats(reset=True, verbose=False)
    t0 = time.perf_counter()
    r = crowd.runSessionGaia(sess, cfg, outDir=os.path.join(OUT, solver), maxOuter=MAX_OUTER,
                             maxInner=MAX_INNER, plotFinal=False, jointSolver=solver)
    wall = time.perf_counter() - t0
    if not r:
        raise RuntimeError(f"the {solver} fit failed -- see the traceback above")
    runs[solver] = next(iter(r.values()))
    timing[solver] = dict(total=wall, **crowd.solverStats(verbose=False))

print()
for s in SOLVERS:
    t = timing[s]
    print(f"{s:7s} total {t['total']:7.1f} s | joint fits {t['seconds']:7.1f} s in {t['calls']} solves "
          f"(largest {t['maxSources']} sources, {t['fallbacks']} fallbacks) | "
          f"{runs[s].summary['nSources']} sources")
old, new = SOLVERS
print(f"\nspeed-up: joint fits {timing[old]['seconds'] / max(timing[new]['seconds'], 1e-9):.1f}x, "
      f"whole run {timing[old]['total'] / timing[new]['total']:.2f}x")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
y = np.arange(len(SOLVERS))
jf = [timing[s]["seconds"] for s in SOLVERS]
rest = [timing[s]["total"] - timing[s]["seconds"] for s in SOLVERS]
ax.barh(y, jf, label="joint flux fits")
ax.barh(y, rest, left=jf, label="everything else", alpha=0.5)
ax.set_yticks(y, SOLVERS)
ax.set_xlabel("seconds")
ax.set_title(f"cutout {CUTOUT}: where the time goes")
ax.legend()
plt.show()

### Do the two runs agree?

Sources are matched by position (within 0.5 px).  For matched sources the PSF magnitudes are
compared with each other and with their errors; unmatched sources are listed with their S/N
(they should be faint, borderline detections).

In [ ]:
pa, pb = runs[old].photometry, runs[new].photometry
xa, xb = np.c_[pa["x"], pa["y"]], np.c_[pb["x"], pb["y"]]
d, j = cKDTree(xb).query(xa, distance_upper_bound=0.5)
m = np.isfinite(d)
print(f"{old}: {len(pa)} sources, {new}: {len(pb)} sources, matched within 0.5 px: {m.sum()}")
print(f"  only in {old}: {len(pa) - m.sum()}, only in {new}: {len(pb) - len(set(j[m]))}")

ma, mb = np.asarray(pa["psfMag"])[m], np.asarray(pb["psfMag"])[j[m]]
err = np.asarray(pa["psfMagErr"])[m]
ok = np.isfinite(ma) & np.isfinite(mb)
dm = mb[ok] - ma[ok]
print(f"  matched psfMag ({new} - {old}): median {np.median(dm):+.2e} mag, "
      f"robust sigma {1.4826 * np.median(np.abs(dm - np.median(dm))):.2e} mag, "
      f"max |dm| {np.max(np.abs(dm)):.2e} mag")
print(f"  |dm| / psfMagErr: median {np.median(np.abs(dm) / err[ok]):.2e}, "
      f"99th percentile {np.percentile(np.abs(dm) / err[ok], 99):.2e}")
print(f"  position difference: median {np.median(d[m]):.1e} px, max {d[m].max():.1e} px")

onlyA = ~m
onlyB = np.setdiff1d(np.arange(len(pb)), j[m])
if onlyA.any():
    print(f"\n  S/N of sources only in {old}:", np.round(np.sort(np.asarray(pa["psfSnr"])[onlyA]), 1))
if len(onlyB):
    print(f"  S/N of sources only in {new}:", np.round(np.sort(np.asarray(pb["psfSnr"])[onlyB]), 1))

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].plot(ma[ok], dm, ".", ms=3)
axs[0].axhline(0, color="k", lw=0.5)
axs[0].set_xlabel(f"psfMag ({old})")
axs[0].set_ylabel(f"psfMag {new} - {old}")
axs[0].set_title("matched sources")
axs[1].plot(ma[ok], dm / err[ok], ".", ms=3)
axs[1].axhline(0, color="k", lw=0.5)
axs[1].set_xlabel(f"psfMag ({old})")
axs[1].set_ylabel("difference / psfMagErr")
axs[1].set_title("difference in units of the error")
plt.tight_layout()
plt.show()

In [ ]:
for s in SOLVERS:
    sm = runs[s].summary
    print(f"{s:7s} residual robust std per outer pass: "
          + ", ".join(f"{v:.3f}" for v in sm["residStdPerOuter"])
          + f" | sources per pass: {sm['nPeaksPerOuter']}")

In [ ]:
for s in SOLVERS:
    print(f"--- jointSolver = {s} ---")
    crowd.plotFinalComparison(runs[s])
    plt.show()

### Residual difference between the two runs

The final model images subtracted from each other, in units of the noise.  With identical
catalogues this is ~0 everywhere; where a borderline source exists in only one run, it shows up
as an isolated PSF-shaped spot.

In [ ]:
ha, hb = runs[old].outerHistory[-1], runs[new].outerHistory[-1]
modA, modB = ha["result"]["modelFull"], hb["result"]["modelFull"]
var = ha["exposure"].variance.array
diff = (modB - modA) / np.sqrt(np.where(var > 0, var, np.nan))
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(diff, origin="lower", cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label=f"(model {new} - model {old}) / sigma")
ax.set_title("model difference")
plt.show()
print(f"|model difference| / sigma: 99th percentile {np.nanpercentile(np.abs(diff), 99):.2e}, "
      f"max {np.nanmax(np.abs(diff)):.2e}")

## 3. (Optional) a few whole tiles with the new solver

The detector runner takes the same option.  This fits a 2 x 2 block of 400 px tiles with the
new solver (set `RUN_TILES = True`), the way `bin.src/crowdDetector.py fit` does on a full
detector.

In [ ]:
RUN_TILES = False
REGION = (800, 1599, 800, 1599)     # detector region: 2 x 2 tiles of 400 px

if RUN_TILES:
    tileOut = os.path.join(OUT, "tiles_eigen")
    crowd.solverStats(reset=True, verbose=False)
    t0 = time.perf_counter()
    crowd.fitTiles(VISIT, DETECTOR, config=crowd.makeConfig(crowd.CROWDED_STELLAR_FIELD_CONFIG),
                   outDir=tileOut, region=REGION, visitImage=visitImage, psfSource=PSF_SOURCE,
                   maxOuter=MAX_OUTER, maxInner=MAX_INNER, jointSolver="eigen")
    print(f"tiles: {time.perf_counter() - t0:.0f} s")
    crowd.solverStats()
    mosaic = crowd.stitchTiles(tileOut)
    crowd.plotDetector(mosaic)
    plt.show()

## Notes

* **What was taken from pipe_crowd:** its `CrowdedFieldMatrix::solve` — the sparse design matrix
  handed to `Eigen::LeastSquaresConjugateGradient` (Jacobi-preconditioned conjugate gradients on
  the normal equations).  meas_crowd already built a sparse matrix with fast C++ PSF stamps, so the
  pipe_crowd matrix *construction* (one `psf.computeImage` per source, `std::map` pixel indexing)
  was not needed.
* **Differences from pipe_crowd:** pipe_crowd's solve is unconstrained; here an active-set loop
  keeps fluxes >= 0 as meas_crowd's fit requires (`crowd.solveSparse(A, b, nonNegative=False)`
  is the plain pipe_crowd solve).  pipe_crowd weights matrix and data by 1/variance; meas_crowd
  keeps its 1/sigma weighting, which is the correct chi^2.  The tolerance is 1e-8 (pipe_crowd:
  1e-6) so fluxes match the exact solvers; change it with `crowd.setJointSolver("eigen", tol=...)`.
* **Safety net:** if a sparse solve does not converge, that fit is redone with the previous
  solver and counted in `solverStats()["fallbacks"]`.
* To go back to the previous behaviour: `jointSolver="normal"` in the config or as a keyword of
  `runSessionGaia` / `fitTiles`, or `--joint-solver normal` for `crowdDetector.py fit`.